In [1]:
import pandas as pd
import numpy as np
from surprise import SVD
from surprise import Dataset
from surprise import Reader
from surprise.model_selection import train_test_split


movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")
print(np.__version__)

print(movies.head())
print(ratings.head())

1.26.4
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [ ]:
# ===== STEP 2: PREPROCESS =====

# Lọc dữ liệu để chỉ giữ lại những người dùng đã đánh giá ít nhất 20 phim
user_counts = ratings['userId'].value_counts()

ratings = ratings[
    ratings['userId'].isin(user_counts[user_counts >= 20].index)
]

print("After filter:", len(ratings))

After filter: 100836


In [3]:
# ===== STEP 3: SPLIT DATA =====
from surprise import Dataset
from surprise import Reader
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
#trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [4]:
# ===== STEP 4: TRAIN DATA =====
from surprise import SVD
from surprise import accuracy
from surprise.model_selection import train_test_split

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
model = SVD(random_state=42)
model.fit(trainset)

In [5]:
# ===== STEP 5: EVALUATE DATA =====
predictions = model.test(testset)
rmse = accuracy.rmse(predictions, verbose=False)
mae = accuracy.mae(predictions, verbose=False)
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

RMSE: 0.8805
RMSE: 0.8805327284055429


In [6]:
user_id = 1
rated_movie_ids = set(ratings.loc[ratings['userId'] == user_id, 'movieId'])
candidate_movie_ids = [movie_id for movie_id in ratings['movieId'].unique() if movie_id not in rated_movie_ids]

predictions = []
for movie_id in candidate_movie_ids:
    pred = model.predict(user_id, movie_id)
    predictions.append((movie_id, pred.est))

predictions.sort(key=lambda x: x[1], reverse=True)
top_10 = predictions[:10]

print(f"Top 10 recommended movies for userId={user_id}:")
for movie_id, rating in top_10:
    movie_title = movies.loc[movies['movieId'] == movie_id, 'title'].values[0]
    print(f"{movie_title}: Predicted Rating = {rating:.2f}")

Top 10 recommended movies for userId=1:
Seven (a.k.a. Se7en) (1995): Predicted Rating = 5.00
Usual Suspects, The (1995): Predicted Rating = 5.00
Goodfellas (1990): Predicted Rating = 5.00
American History X (1998): Predicted Rating = 5.00
Shawshank Redemption, The (1994): Predicted Rating = 5.00
Good Will Hunting (1997): Predicted Rating = 5.00
Departed, The (2006): Predicted Rating = 5.00
Inglourious Basterds (2009): Predicted Rating = 5.00
This Is Spinal Tap (1984): Predicted Rating = 5.00
Life Is Beautiful (La Vita è bella) (1997): Predicted Rating = 5.00
